In [1]:
import numpy as np
import pandas as pd

### Importing Raw Datasets

In [2]:
raw_crime = pd.read_csv(
    "datasets/raw/raw_crime_2024.csv",
    sep=";",
    low_memory=False,
    dtype={"WijkenEnBuurten": "string", "ID": "string"},
)
raw_nbh = pd.read_excel("datasets/raw/raw_nbh_2024.xlsx")
for _geo_id in ("gwb_code_10", "gwb_code_8", "gwb_code"):
    if _geo_id in raw_nbh.columns:
        raw_nbh[_geo_id] = raw_nbh[_geo_id].astype("string")

In [3]:
raw_nbh.shape

(18310, 126)

### Check how many 'BU' for Buurten

In [4]:
print("Crime Cols:", raw_crime.columns)
print("NBH Cols:", raw_nbh.columns)

Crime Cols: Index(['ID', 'SoortMisdrijf', 'WijkenEnBuurten', 'Perioden',
       'GeregistreerdeMisdrijven_1', 'Gemeentenaam_2', 'SoortRegio_3'],
      dtype='object')
NBH Cols: Index(['gwb_code_10', 'gwb_code_8', 'regio', 'gm_naam', 'recs', 'gwb_code',
       'ind_wbi', 'a_inw', 'a_man', 'a_vrouw',
       ...
       'g_afs_kv', 'g_afs_sc', 'g_3km_sc', 'a_opp_ha', 'a_lan_ha', 'a_wat_ha',
       'pst_mvp', 'pst_dekp', 'ste_mvs', 'ste_oad'],
      dtype='object', length=126)


In [19]:
# Print counts of GM/WK/BU for crime dataset
print("GM_crime:", (raw_crime['WijkenEnBuurten'].astype(str).str.startswith('GM')).sum())
print("WK_crime:", (raw_crime['WijkenEnBuurten'].astype(str).str.startswith('WK')).sum())
print("BU_crime:", (raw_crime['WijkenEnBuurten'].astype(str).str.startswith('BU')).sum())
print("Total Crime Rows", raw_crime.shape[0])
print()

# Print counts of GM/WK/BU for neighbourhood dataset
print("GM_nbh:", (raw_nbh['gwb_code_10'].astype(str).str.startswith('GM')).sum())
print("WK_nbh:", (raw_nbh['gwb_code_10'].astype(str).str.startswith('WK')).sum())
print("BU_nbh:", (raw_nbh['gwb_code_10'].astype(str).str.startswith('BU')).sum())
print("Total NBH Rows", raw_nbh.shape[0])


GM_crime: 343
WK_crime: 3424
BU_crime: 14730
Total Crime Rows 18498

GM_nbh: 342
WK_nbh: 3393
BU_nbh: 14574
Total NBH Rows 18310


### Remove 'GM' and 'WK' from both datasets

In [6]:
# Keep only neighborhood (BU) level: drop rows where code starts with 'GM' or 'WK'.
crime = raw_crime[~raw_crime['WijkenEnBuurten'].astype(str).str.strip().str.startswith(('GM', 'WK'), na=False)]
nbh = raw_nbh[~raw_nbh['gwb_code_10'].astype(str).str.strip().str.startswith(('GM', 'WK'), na=False)]


### Remove one remaining row which starts with 'NL'

In [7]:
# drop rows where code starts with 'NL'
crime = crime[~crime['WijkenEnBuurten'].astype(str).str.strip().str.startswith(('NL'), na=False)]
nbh = nbh[~nbh['gwb_code_10'].astype(str).str.strip().str.startswith(('NL'), na=False)]

print("Rows in crime after dropping 'GM', 'WK', and 'NL':", crime.shape[0])
print("Rows in nbh after dropping 'GM', 'WK', and 'NL':", nbh.shape[0])

Rows in crime after dropping 'GM', 'WK', and 'NL': 14730
Rows in nbh after dropping 'GM', 'WK', and 'NL': 14574


## Merging datasets

### Checking missing values for merge column - 0

In [8]:
crime["WijkenEnBuurten"].isna().mean(), nbh["gwb_code_10"].isna().mean()

(np.float64(0.0), np.float64(0.0))

### Standardize & Create join keys

In [9]:
# Standardize join keys (strip whitespace, uppercase, handle missing tokens).

def clean_key(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
        .str.upper()
        .replace({"NAN": np.nan, "NONE": np.nan})
    )

crime = crime.copy()
nbh = nbh.copy()

crime["merge_key"] = clean_key(crime["WijkenEnBuurten"])
nbh["merge_key"] = clean_key(nbh["gwb_code_10"])


### Check for Duplicates

In [10]:
crime_key_dupes = crime["merge_key"].duplicated(keep=False).mean()
nbh_key_dupes   = nbh["merge_key"].duplicated(keep=False).mean()

crime_key_dupes, nbh_key_dupes

(np.float64(0.0), np.float64(0.0))

In [11]:
crime["merge_key"].value_counts().head(10)
nbh["merge_key"].value_counts().head(10)

merge_key
BU00140000    1
BU09831307    1
BU09810005    1
BU09810006    1
BU09810007    1
BU09810008    1
BU09810009    1
BU09810100    1
BU09831101    1
BU09831102    1
Name: count, dtype: int64

### Merging

In [12]:
merged = nbh.merge(
    crime.drop(columns=["WijkenEnBuurten"], errors="ignore"),
    on="merge_key",
    how="left",
    validate="one_to_one"
)

merged.shape

(14574, 133)

### Check merging percentage

In [13]:
match_rate = merged["GeregistreerdeMisdrijven_1"].notna().mean()
match_rate

np.float64(0.9945107726087553)

### Remove unmerged rows — missing registered crimes

Rows where **`GeregistreerdeMisdrijven_1`** is missing are treated as neighbourhoods **without a matched crime record** for this merge (not imputed to zero) and are **dropped** so the analysis uses only matched crime counts.

In [14]:
crime_target_col = "GeregistreerdeMisdrijven_1"
missing_crime_mask = merged[crime_target_col].isna()
unmatched = merged.loc[missing_crime_mask, "merge_key"]

# Drop neighbourhoods with no matched crime count (unmatched / missing target)
merged = merged[merged[crime_target_col].notna()].copy().reset_index(drop=True)

In [15]:
merged.shape

(14494, 133)

### Merge neighborhood geometry (GeoPackage)

Join official CBS / PDOK **wijkenbuurten** polygons on the same `merge_key` as above (`buurtcode` standardized ↔ `gwb_code_10`). The CSV export in the next section stays tabular-only; geometry is written to a separate GeoPackage for downstream use (e.g. notebook 4 pre-processing).

In [16]:
import geopandas as gpd

# --- Edit paths if your file lives elsewhere ---
GPKG_PATH = "datasets/raw/wijkenbuurten_2024.gpkg"
GPKG_LAYER = "buurten"  # layer name when the file has multiple layers (e.g. buurten, wijken, gemeenten)
GEO_KEY_COL = "buurtcode"

geo = gpd.read_file(GPKG_PATH, layer=GPKG_LAYER)
geo[GEO_KEY_COL] = geo[GEO_KEY_COL].astype("string")
geo_small = geo[[GEO_KEY_COL, "geometry"]].copy()
geo_small["merge_key"] = clean_key(geo_small[GEO_KEY_COL])

merged_geom = merged.merge(
    geo_small[["merge_key", "geometry"]],
    on="merge_key",
    how="left",
)
merged_gdf = gpd.GeoDataFrame(merged_geom, geometry="geometry", crs=geo.crs)

n = len(merged_gdf)
matched = merged_gdf.geometry.notna().sum()
print(f"Geometry rows matched: {matched} / {n} ({100 * matched / n:.2f}%)")

Geometry rows matched: 14494 / 14494 (100.00%)


### Export final merged dataset

In [17]:
from pathlib import Path

PREPROC_DIR = Path("datasets/pre-processing")
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

final = merged.copy()

out_path = PREPROC_DIR / "merged_crime_nbh_2024.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
final.to_csv(out_path, index=False)
str(out_path)

'datasets/pre-processing/merged_crime_nbh_2024.csv'

### Export merged dataset with geometry (GeoPackage)

Notebook 4 attaches these polygons via this file instead of merging the CBS source `.gpkg` again.

In [18]:
from pathlib import Path

PREPROC_DIR = Path("datasets/pre-processing")
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

gpkg_out = PREPROC_DIR / "merged_crime_nbh_2024.gpkg"
gpkg_out.parent.mkdir(parents=True, exist_ok=True)
merged_gdf.to_file(gpkg_out, driver="GPKG")
str(gpkg_out)

'datasets/pre-processing/merged_crime_nbh_2024.gpkg'